<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Bandits

**[Bandits](https://learning.nextia-ai.com/courses/experiments/m08/bandits/)** · Experiments and Causal Impact: Did Our Change Work? · Module 8, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** write Thompson sampling in a few lines, run it against a fixed equal split on four email subject lines with known click rates, and count the clicks that each way loses.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. The emails are simulated in the notebook. |
| **You should know** | An A/B test and its interval, from [The result and its uncertainty](https://learning.nextia-ai.com/courses/experiments/m03/the-result-and-its-uncertainty/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m08/bandits/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M08-L02-bandits/bandits-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: the project folder

This notebook makes its own data with small simulations, so it downloads nothing. Run the next cell. It makes a folder `larkfield-experiments` and works in it.

You should see `Ready: 0 files in data/.`

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Four subject lines and Thompson sampling

The true click rates are known, because we simulate the clicks. Line C is the best. Thompson sampling keeps clicks and emails sent per line. For each email, it draws one believable click rate per line from a Beta distribution, and sends the line with the highest draw.

In [ ]:
RATES = np.array([0.040, 0.046, 0.052, 0.044])   # lines A, B, C, D (simulated)
NAMES = ["A", "B", "C", "D"]
EMAILS = 40_000

def thompson(clicks, shown, rng):
    """One believable click rate per line, drawn from what we know; the line with the highest draw."""
    draws = rng.beta(1 + clicks, 1 + shown - clicks)
    return int(draws.argmax())

def campaign(policy, seed=40, rates=RATES, emails=EMAILS):
    """Send the emails one by one. Returns emails sent and clicks per line, and the line of each email."""
    rng = np.random.default_rng(seed)
    clicks, shown = np.zeros(len(rates)), np.zeros(len(rates))
    lines = np.empty(emails, dtype=int)
    for i in range(emails):
        line = thompson(clicks, shown, rng) if policy == "thompson" else int(rng.integers(len(rates)))
        shown[line] += 1
        clicks[line] += rng.random() < rates[line]
        lines[i] = line
    return shown, clicks, lines

print("best possible: always line C,", round(RATES.max() * EMAILS), "expected clicks")

> **Check.** You should see `best possible: always line C, 2080 expected clicks`.

## 2. A fixed split against Thompson sampling

In [ ]:
shown_f, clicks_f, lines_f = campaign("fixed")
shown_t, clicks_t, lines_t = campaign("thompson")
for label, shown, clicks in (("fixed split", shown_f, clicks_f), ("Thompson", shown_t, clicks_t)):
    print(f"{label:<12} clicks {int(clicks.sum()):,}; emails per line:",
          ", ".join(f"{n} {int(s):,}" for n, s in zip(NAMES, shown)))

> **Check.** You should see `fixed split  clicks 1,847` and `Thompson     clicks 1,967`. Thompson sampling sent 26,896 emails with line C.

## 3. Where did the emails go?

The share of emails that line C got, in each tenth of the campaign.

In [ ]:
import matplotlib.pyplot as plt
tenths = np.arange(1, 11)
share = lambda lines: [100 * (part == 2).mean() for part in np.array_split(lines, 10)]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(tenths, share(lines_f), "o--", label="fixed split")
ax.plot(tenths, share(lines_t), "o-", label="Thompson sampling")
ax.set_xlabel("Tenth of the 40,000 emails"); ax.set_ylabel("Share of emails to line C (%)"); ax.set_ylim(0, 100)
ax.legend(); plt.show()
print("Thompson, last tenth:", round(share(lines_t)[-1], 1), "%")

> **Check.** You should see a flat line near 25% for the fixed split, a line that rises to near 100% for Thompson sampling, and `Thompson, last tenth: 89.0 %`.

## 4. Your turn: the clicks lost

The **expected clicks lost** are the clicks of always sending line C, minus the clicks that each line's emails are expected to get: `RATES.max() * EMAILS - (shown * RATES).sum()`. Expected clicks use the true rates, so luck in the actual clicks does not count.

> **Your turn.** Compute the expected clicks lost by Thompson sampling, from `shown_t`. Round to 1 decimal and put it in `lost_thompson`. Then compute the same for the fixed split, and compare.

In [ ]:
lost_thompson = round(float(RATES.max() * EMAILS - (shown_t * RATES).sum()), 1)
lost_fixed = round(float(RATES.max() * EMAILS - (shown_f * RATES).sum()), 1)
print(lost_thompson, lost_fixed)

In [ ]:
expect('lost_thompson', lost_thompson, 97.6)

Thompson sampling loses far fewer clicks. Now compare line C with line A under each way: Thompson sampling sent only a few hundred emails with line A, so it knows A's rate badly. A bandit finds the best line; it does not measure the differences well.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m08/bandits/) shows what a bandit cannot tell you, and has the knowledge checks that count toward your certificate. In the project: `xp bandit` and `xp bandit --repeats 100`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Report a result people trust](https://learning.nextia-ai.com/courses/experiments/m08/report-a-result-people-trust/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m08/bandits/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.